# Archivar 3D Visualization - Getting Started

This notebook demonstrates how to:
1. Load a DXF part file
2. Convert it to 3D geometry
3. View it interactively

## Setup

Make sure you've installed the package:
```bash
cd /path/to/archivar
pip install -e ".[dev]"
```

In [ ]:
# First, let's verify the installation
import sys
print(f"Python: {sys.version}")

import ezdxf
print(f"ezdxf: {ezdxf.__version__}")

import build123d
print(f"build123d: {build123d.__version__}")

print("\nAll dependencies installed!")

In [ ]:
# Set up the 3D viewer for Jupyter
from jupyter_cadquery import show, set_defaults
from jupyter_cadquery.viewer.client import is_cq_jupyter_viewer_running

# Configure viewer defaults
set_defaults(
    theme="light",
    axes=True,
    axes0=True,
    grid=(True, True, False),
    transparent=False,
    default_color=(0.8, 0.7, 0.5),  # Plywood-ish color
)

print("Viewer configured!")

## Load a DXF Part

Let's start with a simple part - one of the side plates.

In [ ]:
from pathlib import Path
import os

# Find the CAD directory (works whether we're in notebooks/ or project root)
notebook_dir = Path.cwd()
if notebook_dir.name == "notebooks":
    project_root = notebook_dir.parent
else:
    project_root = notebook_dir

cad_dir = project_root / "CAD"
individual_parts = cad_dir / "individual_parts"

print(f"Project root: {project_root}")
print(f"CAD directory: {cad_dir}")
print(f"\nAvailable part categories:")
for d in sorted(individual_parts.iterdir()):
    if d.is_dir():
        print(f"  - {d.name}")

In [ ]:
# List parts in the main frame category
main_frame_dir = individual_parts / "01_main_frame" / "18mm"

print("Main frame parts (18mm):")
for f in sorted(main_frame_dir.glob("*.dxf")):
    print(f"  - {f.name}")

In [ ]:
# Load the side plate DXF
from archivar.dxf import load_dxf, dxf_to_face, extrude_dxf

side_plate_path = main_frame_dir / "side_plate.dxf"
print(f"Loading: {side_plate_path}")

doc = load_dxf(side_plate_path)
print(f"DXF version: {doc.dxfversion}")
print(f"Entities in modelspace: {len(list(doc.modelspace()))}")

## Convert to 3D

Now let's extrude the 2D profile into a 3D solid. The main frame parts are 18mm thick plywood.

In [ ]:
# Create a 3D solid by extruding the 2D profile
THICKNESS_18MM = 18.0  # mm

side_plate_3d = extrude_dxf(side_plate_path, THICKNESS_18MM)
print(f"Created 3D solid!")
print(f"Volume: {side_plate_3d.volume:.1f} mm³")

In [ ]:
# Display the 3D part!
show(side_plate_3d)

## Try Another Part

Let's load a different part to make sure our loader works generally.

In [ ]:
# Load the front plate
front_plate_path = main_frame_dir / "front_plate.dxf"

if front_plate_path.exists():
    front_plate_3d = extrude_dxf(front_plate_path, THICKNESS_18MM)
    print(f"Front plate volume: {front_plate_3d.volume:.1f} mm³")
    show(front_plate_3d)
else:
    print(f"File not found: {front_plate_path}")
    print("Available files:")
    for f in main_frame_dir.glob("*.dxf"):
        print(f"  {f.name}")

## Next Steps

Now that we can load individual parts, the next notebooks will cover:

- **02_viewing_parts.ipynb**: Browse all parts, organized by category
- **03_assembly.ipynb**: Position parts in 3D space to see the full assembly

### Quick Reference

Material thicknesses in this project:
- Main frame: 18mm (made from 2x 9mm glued)
- Lighting module: 4mm (3.2mm actual from SendCutSend)